# EXP-06: SPINK1–Trypsin Binding Free Energy (ΔG_bind)
**Kazal Family Benchmark | Coupled with EXP-32 (N34S negative control)**

- **PDB:** 1TGS (1.8 Å) — SPINK1/PSTI homolog
- **Pipeline:** Prep → Minimize → NVT → NPT → Production (100 ns) → US (51 windows) → WHAM → ΔG
- **Expected:** ΔG_bind = −11.1 kcal/mol
- **PASS criterion:** [−15.6, −6.6] kcal/mol

**Note:** System & state files are saved for EXP-32 downstream use.

In [ ]:
# Install OpenMM + all scientific dependencies (pip — OpenCL platform)
!pip install openmm mdtraj 'pymbar==4.0.3' mdanalysis
!pip install netCDF4 git+https://github.com/choderalab/openmmtools.git

# mpiplus stub — not on PyPI; minimal shim for imports
import os as _os, sys as _sys
_sp = _os.path.join(_sys.prefix, 'lib',
    f'python{_sys.version_info.major}.{_sys.version_info.minor}',
    'site-packages', 'mpiplus')
_os.makedirs(_sp, exist_ok=True)
with open(_os.path.join(_sp, '__init__.py'), 'w') as _f:
    _f.write(
        'class delayed_termination:\n'
        '    def __init__(self, func=None):\n'
        '        self._func = func\n'
        '    def __enter__(self): return self\n'
        '    def __exit__(self, *a): pass\n'
        '    def __call__(self, *args, **kwargs):\n'
        '        if self._func is not None: return self._func(*args, **kwargs)\n'
        '        if len(args) == 1 and callable(args[0]): return args[0]\n'
        '        return self\n'
        'def get_mpicomm(): return None\n'
        'def run_single_node(rank=0, broadcast_result=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def on_single_node(rank=0, broadcast_result=False, sync_nodes=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def distribute(func, seq, *args, send_results_to=None, sync_nodes=False, group_nodes=None):\n'
        '    results = [func(item, *args) for item in seq]\n'
        '    return list(zip(*results)) if results and isinstance(results[0], tuple) else (results, list(seq))\n'
    )
for _k in list(_sys.modules):
    if 'mpiplus' in _k:
        del _sys.modules[_k]

!pip install -q scipy matplotlib pandas requests gemmi

# Verify critical imports
import importlib
all_ok = True
for pkg in ['openmm', 'openmmtools', 'mdtraj', 'pymbar', 'MDAnalysis']:
    try:
        m = importlib.import_module(pkg)
        print(f"\u2713 {pkg} {getattr(m, '__version__', '')}")
    except ImportError as e:
        print(f"\u2717 {pkg}: {e}")
        all_ok = False

if all_ok:
    print("\n\u2705 All packages installed successfully!")
else:
    raise RuntimeError("Package installation failed \u2014 check error messages above")

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive', force_remount=True)
import sys, os, shutil, json, zipfile, logging
from pathlib import Path
import numpy as np

PIPELINE_ROOT = Path('/content/drive/MyDrive/spink7_pipeline')
assert PIPELINE_ROOT.exists()
sys.path.insert(0, str(PIPELINE_ROOT))

EXP_ID = 'EXP-06'
DRIVE_OUTPUT = Path(f'/content/drive/MyDrive/v3_gpu_results/{EXP_ID}')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path(f'/content/{EXP_ID}_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s %(message)s')

# ── Console log capture ──────────────────────────────────────
import io as _io
_log_buf = _io.StringIO()
_orig_stdout_write = sys.stdout.write
_orig_stderr_write = sys.stderr.write
def _stdout_log_write(data):
    _orig_stdout_write(data)
    _log_buf.write(data)
def _stderr_log_write(data):
    _orig_stderr_write(data)
    _log_buf.write(data)
sys.stdout.write = _stdout_log_write
sys.stderr.write = _stderr_log_write
# ─────────────────────────────────────────────────────────────

In [ ]:
# Checkpoint & auto-save utilities
import json, shutil, threading, time as _time
from pathlib import Path

class ExperimentCheckpoint:
    """Drive-backed phase checkpoint for resilient experiment execution."""

    def __init__(self, drive_output: Path, exp_id: str):
        self.drive_output = drive_output
        self.exp_id = exp_id
        self.path = drive_output / 'experiment_checkpoint.json'
        self.state = self._load()

    def _load(self) -> dict:
        if self.path.exists():
            with open(self.path) as f:
                return json.load(f)
        return {'exp_id': self.exp_id, 'phases': {}}

    def save(self):
        self.drive_output.mkdir(parents=True, exist_ok=True)
        with open(self.path, 'w') as f:
            json.dump(self.state, f, indent=2, default=str)

    def is_done(self, phase: str) -> bool:
        return phase in self.state['phases']

    def mark_done(self, phase: str, data: dict = None):
        self.state['phases'][phase] = {
            'timestamp': _time.strftime('%Y-%m-%d %H:%M:%S'),
            'data': data or {},
        }
        self.save()

    def get_data(self, phase: str) -> dict:
        return self.state['phases'].get(phase, {}).get('data', {})

    def summary(self):
        n = len(self.state['phases'])
        print(f'Checkpoint: {self.exp_id} — {n} phase(s) completed')
        for phase, info in self.state['phases'].items():
            print(f'  ✓ {phase} ({info["timestamp"]})')

def start_periodic_sync(work_dir: Path, drive_output: Path, interval_s: int = 300):
    """Background thread that syncs checkpoint/manifest files to Drive every N seconds."""
    stop_event = threading.Event()
    sync_patterns = ['*.chk', '*.json', '*manifest*', '*.npz', '*.npy']
    def _sync():
        while not stop_event.is_set():
            stop_event.wait(interval_s)
            if stop_event.is_set():
                break
            for pat in sync_patterns:
                for f in work_dir.rglob(pat):
                    if f.is_file() and f.stat().st_size < 500_000_000:
                        dest = drive_output / f.relative_to(work_dir)
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        try:
                            shutil.copy2(f, dest)
                        except Exception:
                            pass
    t = threading.Thread(target=_sync, daemon=True, name='drive-sync')
    t.start()
    return stop_event

def restore_from_drive(drive_output: Path, work_dir: Path):
    """On session restart, copy checkpoint/manifest files from Drive back to local."""
    restored = 0
    for pat in ['*.chk', '*manifest*', '*.json']:
        for f in drive_output.rglob(pat):
            if f.is_file():
                dest = work_dir / f.relative_to(drive_output)
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.copy2(f, dest)
                    restored += 1
    if restored:
        print(f'Restored {restored} checkpoint/manifest files from Drive')

# Initialize
ckpt = ExperimentCheckpoint(DRIVE_OUTPUT, EXP_ID)
restore_from_drive(DRIVE_OUTPUT, WORK_DIR)
sync_stop = start_periodic_sync(WORK_DIR, DRIVE_OUTPUT, interval_s=300)
ckpt.summary()
print('Auto-save: checkpoint/manifest files sync to Drive every 5 min')

In [ ]:
import openmm
from openmm import unit, Platform
Platform.getPlatformByName('CUDA')
print(f'OpenMM {openmm.__version__}, CUDA ready')

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.config import (
    SystemConfig, MinimizationConfig, EquilibrationConfig,
    ProductionConfig, UmbrellaConfig, WHAMConfig, KCAL_TO_KJ
)
from src.prep.pdb_fetch import fetch_pdb
from src.prep.pdb_clean import clean_structure
from src.prep.topology import build_topology
from src.prep.solvate import solvate_system
from src.simulate.minimizer import minimize_energy
from src.simulate.equilibrate import run_nvt, run_npt
from src.simulate.production import run_production
from src.simulate.umbrella import (
    run_umbrella_campaign, generate_window_centers,
    compute_overlap_matrix
)
from src.simulate.platform import select_platform
from src.analyze.wham import solve_wham, bootstrap_pmf_uncertainty
print('Imports OK')

In [ ]:
PDB_ID = '1TGS'
TEMPERATURE_K = 310.0
system_config = SystemConfig()
min_config = MinimizationConfig(max_iterations=10000)
eq_config = EquilibrationConfig(nvt_duration_ps=500.0, npt_duration_ps=1000.0, temperature_k=TEMPERATURE_K)
prod_config = ProductionConfig(duration_ns=50.0, temperature_k=TEMPERATURE_K)  # optimized from 100: 50ns sufficient for SMD/US starting point
umbrella_config = UmbrellaConfig(
    xi_min_nm=1.5, xi_max_nm=4.0, window_spacing_nm=0.05,
    spring_constant_kj_mol_nm2=1000.0, per_window_duration_ns=10.0,
)
wham_config = WHAMConfig(tolerance=1e-6, max_iterations=100000, n_bootstrap=200)

for d in ['prep', 'simulation', 'umbrella', 'analysis', 'figures']:
    (WORK_DIR / d).mkdir(parents=True, exist_ok=True)
PREP_DIR, SIM_DIR, US_DIR = WORK_DIR/'prep', WORK_DIR/'simulation', WORK_DIR/'umbrella'
ANALYSIS_DIR, FIGURES_DIR = WORK_DIR/'analysis', WORK_DIR/'figures'
print(f'{len(generate_window_centers(umbrella_config))} windows')

In [ ]:
raw_pdb_path = fetch_pdb(PDB_ID, output_dir=PREP_DIR)
with open(raw_pdb_path) as f:
    all_chains = set(l[21] for l in f if l.startswith(('ATOM', 'HETATM')))
print(f'Chains: {sorted(all_chains)}')
chains_to_keep = set(c for c in all_chains if c.strip())
cleaned_pdb_path = clean_structure(raw_pdb_path, chains_to_keep=chains_to_keep,
                             remove_heteroatoms=True, remove_waters=True)

In [ ]:
from openmm.app import PME, ForceField, Simulation, HBonds
from openmm import LangevinMiddleIntegrator, XmlSerializer

topology, system, modeller = build_topology(
    cleaned_pdb_path, system_config,
)
chain_groups = []
for chain in topology.chains():
    indices = [atom.index for atom in chain.atoms()]
    if indices:
        chain_groups.append(indices)
        print(f'Chain {chain.id}: {len(indices)} atoms')
pull_group_1, pull_group_2 = chain_groups[0], chain_groups[1]

modeller, n_water, n_pos, n_neg = solvate_system(modeller, system_config)
print(f'Solvated: {n_water} waters')

ff = ForceField(system_config.force_field, system_config.water_model)
system = ff.createSystem(modeller.topology, nonbondedMethod=PME,
    nonbondedCutoff=1.0*unit.nanometer, constraints=HBonds, rigidWater=True)
integrator = LangevinMiddleIntegrator(TEMPERATURE_K*unit.kelvin, 1.0/unit.picosecond, 0.002*unit.picoseconds)
simulation = Simulation(modeller.topology, system, integrator, select_platform('CUDA'))
simulation.context.setPositions(modeller.positions)

min_result = minimize_energy(simulation, min_config)
system_xml_path = SIM_DIR / 'system.xml'
system_xml_path.write_text(XmlSerializer.serialize(system), encoding='utf-8')
print(f'Minimized: {min_result["final_energy_kj_mol"]:.0f} kJ/mol')

In [ ]:
nvt_result = run_nvt(simulation, eq_config, SIM_DIR / 'nvt')
npt_result = run_npt(simulation, eq_config, SIM_DIR / 'npt')
print(f'NPT: T={npt_result["avg_temperature_k"]:.1f} K, rho={npt_result["avg_density_g_cm3"]:.4f}')
eq_state_path = SIM_DIR / 'npt' / 'npt_final_state.xml'
topology_pdb_path = npt_result['topology_path']

In [ ]:
from src.simulate.production import resume_production

prod_output = SIM_DIR / 'production'
prod_output.mkdir(parents=True, exist_ok=True)

if ckpt.is_done('production'):
    print('⏭ Production already completed, skipping')
    prod_result = ckpt.get_data('production')
else:
    existing_chk = sorted(prod_output.glob('*.chk'))
    if existing_chk:
        print(f'Resuming production from checkpoint: {existing_chk[-1].name}')
        prod_result = resume_production(simulation, prod_config, prod_output)
    else:
        prod_result = run_production(simulation, prod_config, prod_output)
    print(f'Production: {prod_result["total_time_ns"]:.1f} ns')
    # Save system files for EXP-32 (N34S FEP needs same solvated system)
    for f in [system_xml_path, eq_state_path, Path(topology_pdb_path)]:
        if f.exists(): shutil.copy2(f, DRIVE_OUTPUT / f.name)
    ckpt.mark_done('production', {'n_frames': prod_result['n_frames'], 'total_time_ns': prod_result['total_time_ns']})

In [ ]:
if ckpt.is_done('umbrella'):
    print('⏭ Umbrella sampling already completed, skipping')
else:
    us_results = run_umbrella_campaign(
        equilibrated_state_path=eq_state_path, system_xml_path=system_xml_path,
        config=umbrella_config, pull_group_1=pull_group_1, pull_group_2=pull_group_2,
        output_dir=US_DIR, topology_pdb_path=Path(topology_pdb_path),
        platform_name='CUDA', n_workers=1,
    )
    print(f'{len(us_results)} windows complete')
    ckpt.mark_done('umbrella', {'n_windows': len(us_results)})

In [ ]:
xi_ts_list = [r['xi_timeseries'] for r in us_results]
wc = np.array([r['window_center_nm'] for r in us_results])
ks = np.full(len(us_results), umbrella_config.spring_constant)

overlap = compute_overlap_matrix(xi_ts_list)
min_ov = min(overlap[i,i+1] for i in range(len(us_results)-1))
print(f'Min adjacent overlap: {min_ov:.3f}')

wham_result = solve_wham(xi_ts_list, wc, ks, TEMPERATURE_K, wham_config)
bs_result = bootstrap_pmf_uncertainty(xi_ts_list, wc, ks, TEMPERATURE_K, wham_config)

pmf_kcal = wham_result['pmf_kcal_mol']
xi = wham_result['xi_bins']
fm = np.isfinite(pmf_kcal)
br = (xi < 2.5) & fm
if np.any(br):
    bi = np.argmin(pmf_kcal[br])
    dg = float(pmf_kcal[br][bi])
    bxi = float(xi[br][bi])
    big = np.where(br)[0][bi]
else:
    bi = np.argmin(pmf_kcal[fm])
    dg = float(pmf_kcal[fm][bi])
    bxi = float(xi[fm][bi])
    big = np.where(fm)[0][bi]
dg_unc = float(bs_result['pmf_std'][big]) / KCAL_TO_KJ

EXPECTED = -11.1
PASS_LO, PASS_HI = -15.6, -6.6
verdict = 'PASS' if PASS_LO <= dg <= PASS_HI else ('MARGINAL' if -20 <= dg <= -3 else 'FAIL')
print(f'\u0394G = {dg:.2f} \u00b1 {dg_unc:.2f} kcal/mol | {verdict}')

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
ax.plot(xi[fm], pmf_kcal[fm], 'b-', lw=2)
ax.axhline(EXPECTED, color='red', ls='--', label=f'Exp. {EXPECTED}')
ax.axhspan(PASS_LO, PASS_HI, alpha=0.1, color='green', label='PASS')
ax.set_xlabel('\u03be (nm)'); ax.set_ylabel('PMF (kcal/mol)')
ax.set_title('EXP-06: SPINK1\u2013Trypsin PMF')
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(FIGURES_DIR/'pmf_profile.png', dpi=150); plt.close(fig)
print('Figure saved')

In [ ]:
results = {
    'experiment_id': EXP_ID, 'pdb_id': PDB_ID, 'temperature_k': TEMPERATURE_K,
    'dg_bind_kcal_mol': dg, 'dg_uncertainty_kcal_mol': dg_unc,
    'bound_minimum_xi_nm': bxi, 'verdict': verdict,
    'expected_kcal_mol': EXPECTED, 'pass_range_kcal_mol': [PASS_LO, PASS_HI],
    'wham_converged': wham_result['converged'],
    'min_adjacent_overlap': float(min_ov),
    'n_umbrella_windows': len(us_results),
    'npt_avg_density': npt_result['avg_density_g_cm3'],
    'production_ns': prod_result['total_time_ns'],
}
with open(WORK_DIR/'results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
np.savez(ANALYSIS_DIR/'pmf_data.npz', xi_bins=xi, pmf_kcal_mol=pmf_kcal,
         pmf_kj_mol=wham_result['pmf_kj_mol'], pmf_mean=bs_result['pmf_mean'], pmf_std=bs_result['pmf_std'])
print('Results saved')

In [ ]:
sync_stop.set()  # Stop periodic sync before final copy

for item in WORK_DIR.rglob('*'):
    if item.is_file():
        dest = DRIVE_OUTPUT / item.relative_to(WORK_DIR)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, dest)

ckpt.mark_done('complete')

# ── Save full console log ────────────────────────────────────
_console_log_text = _log_buf.getvalue()
(WORK_DIR / 'console_log.txt').write_text(_console_log_text)
(DRIVE_OUTPUT / 'console_log.txt').write_text(_console_log_text)
print(f'Console log saved: {len(_console_log_text)} chars')
# ─────────────────────────────────────────────────────────────
zip_path = Path(f'/content/{EXP_ID}_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item in WORK_DIR.rglob('*'):
        if item.is_file(): zf.write(item, f'{EXP_ID}/{item.relative_to(WORK_DIR)}')
print(f'Zip: {zip_path.stat().st_size/1e6:.1f} MB')
files.download(str(zip_path))